# Analysis of Oscar Nominated Movies 2000–2017
#### Fall 2024 Data Science Project
Jack Burke – A, C, F

Tal Ledeniov – A, C, D, E

Ryan DeCamp – A, C, F

Robert Hopkins – A, B, C, G

## Introduction

The objective of this project is to guide readers through the data science lifecycle, as we analyze ....

Through this tutorial, readers should learn enough to complete each step of the data science life cycle for *their own* problem they'd like to solve. The steps in the data science lifecycle are as follows:
- Data collection
- Data processing
- Exploratory analysis & visualization
- Analysis, hypothesis testing, & ML
- Insight & decision making

## Data Curation

The first step is of course collecting data. Data can either be collected firsthand directly from a source (primary) or can involve use of pre-existing sources that have already been collected and published for public use (secondary). For this project, we'll be using *secondary* data collection, which is less costly in terms of time and money. 

The data that we'll be using for our tutorial today is from [Kaggle](https://kaggle.com), a great place to find tons of data on tons of different topics. The dataset we'll use is [Oscar nominated Movies 2000–2017](https://www.kaggle.com/datasets/vipulgote4/oscars-nominated-movies-from-2000-to-2017), published by user *Overfitted*, sourced from [IMDB.com](https://IMDB.com)

Kaggle unfortunately requires users to create an account to download data, so the CSV is provided [(download link)](oscar_movies.csv) for this tutorial.

Now, let's get this into a Pandas *DataFrame* so we can work with the data in Python! 

In [23]:
# First, we'll import the packages we need for now and for later. 
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

The packages imported above are so fundamental they break the general Python style rule that says you shouldn't use `import ... as ...`. People know what `pd`, `np`, and the others are, so it's OK! 

In [35]:
# import from local file system. 
# (assumes that your .py or .ipynb file is in the same folder as the .csv file.)
movie_df = pd.read_csv("./oscar_movies.csv")
movie_df.head() # DataFrame.head() shows the first few rows of the dataframe.

,year,movie,movie_id,certificate,duration,genre,rate,metascore,synopsis,votes,...,New_York_Film_Critics_Circle_nominated,New_York_Film_Critics_Circle_nominated_categories,Los_Angeles_Film_Critics_Association_won,Los_Angeles_Film_Critics_Association_won_categories,Los_Angeles_Film_Critics_Association_nominated,Los_Angeles_Film_Critics_Association_nominated_categories,release_date.year,release_date.month,release_date.day-of-month,release_date.day-of-week
0,2001,Kate & Leopold,tt0035423,PG-13,118,Comedy|Fantasy|Romance,6.4,44.0,An English Duke from 1876 is inadvertedly drag...,66660,...,0,NaN,0,NaN,0,NaN,2001.0,12.0,25.0,2.0
1,2000,Chicken Run,tt0120630,G,84,Animation|Adventure|Comedy,7.0,88.0,When a cockerel apparently flies into a chicke...,144475,...,1,Best Animated Film,1,Best Animation,1,Best Animation,2000.0,6.0,23.0,5.0
2,2005,Fantastic Four,tt0120667,PG-13,106,Action|Adventure|Family,5.7,40.0,A group of astronauts gain superpowers after a...,273203,...,0,NaN,0,NaN,0,NaN,2005.0,7.0,8.0,5.0
3,2002,Frida,tt0120679,R,123,Biography|Drama|Romance,7.4,61.0,"A biography of artist Frida Kahlo, who channel...",63852,...,0,NaN,0,NaN,0,NaN,2002.0,11.0,22.0,5.0
4,2001,The Lord of the Rings: The Fellowship of the Ring,tt0120737,PG-13,178,Adventure|Drama|Fantasy,8.8,92.0,A meek Hobbit from the Shire and eight compani...,1286275,...,0,NaN,1,Best Music,2,Best Music|Best Production Design,2001.0,12.0,19.0,3.0


Now that we have our data *directly* from the Kaggle source, we can get to some *real* data science.

### Data Processing

Actually, we need to prepare our data before we can analyze it. The purpose of this is to make our data easy to work with and to remedy any errors that may or may not have seeped in. The last thing we want is to draw faulty conclusions from faulty data. Let's get cleaning.

After making sure that we don't have any duplicate entries, we'll go through and change some data types (and perform other small, quick fixes) to make things easy to work with. 
- Convert flags to booleans
- Metascore to int
- Convert dates to datetime types
- Re-align day-of-week to be 0-indexed
- Convert "release_date.____" columns to int
- Convert lists to list types (genre, _categories)

In [36]:
assert len(movie_df[movie_df.duplicated()]) == 0 # Make sure all IDs are unique

# This shows us movies with the same names. 
# Movies are allowed to have the same name, so these are OK. 
movie_df[movie_df.duplicated("movie", keep=False)]

,year,movie,movie_id,certificate,duration,genre,rate,metascore,synopsis,votes,...,New_York_Film_Critics_Circle_nominated,New_York_Film_Critics_Circle_nominated_categories,Los_Angeles_Film_Critics_Association_won,Los_Angeles_Film_Critics_Association_won_categories,Los_Angeles_Film_Critics_Association_nominated,Los_Angeles_Film_Critics_Association_nominated_categories,release_date.year,release_date.month,release_date.day-of-month,release_date.day-of-week
484,2006,The Illusionist,tt0443543,PG-13,110,Drama|Mystery|Romance,7.6,68.0,"In turn-of-the-century Vienna, a magician uses...",304063,...,0,NaN,0,NaN,0,NaN,2006.0,9.0,1.0,5.0
588,2010,The Illusionist,tt0775489,PG,80,Animation|Drama,7.5,82.0,A French illusionist finds himself out of work...,29236,...,1,Best Animated Film,0,NaN,1,Best Animation,2011.0,2.0,11.0,5.0


Oscars, as the most important prestigious type of award for films, and the focus of this dataset, are *each* represented for each movie with a unique flag. That means that movie XXXX has a series of "Yes" and "No" for Best Picture, Best International, .... We want it to be boolean types, not strings.

In [37]:
# Convert Oscar flags to bools instead of "Yes", "No" strings. 
for column in movie_df.columns:
    # get relevant Oscar flag columns
    if column.startswith("Oscar_") and column not in ["Oscar_nominated", "Oscar_nominated_categories"]:
        movie_df[column] = movie_df[column].replace({"Yes": True, "No": False})

movie_df["Oscar_Best_Picture_won"].dtype

dtype('bool')

Metascore is an integer, but the column is a float! Let's fix it.

In [ ]:
# Convert metascore to int.
movie_df["metascore"] = pd.to_numeric(movie_df["metascore"], errors='coerce').astype('Int64')

movie_df[movie_df["metascore"].isna()][["movie"]] # 13 movies are missing metascores. Unpopular movies. 

,movie
88,Ken Park
272,Twin Sisters
348,The Room
380,As It Is in Heaven
450,Yesterday
474,The Queen
577,Beaufort
652,Unthinkable
669,"Angus, Thongs and Perfect Snogging"
700,Death Proof


Datetime types are provided by the datetime module, and pandas gives us an easy way to convert to datetime type. The type lets us do convenient things with dates that would be more difficult as a string or a series of numbers, so it's typical to use it.

In [39]:
# Convert "release_date" column to datetime types
movie_df["release_date"] = pd.to_datetime(movie_df["release_date"], dayfirst=False)

movie_df["release_date"].dtype
# dtype('<M8[ns]') is a datetype type

dtype('<M8[ns]')

In [40]:
# Align "release_date.day-of-week", the following is what we want.
"""
0: mon
1: tues
2: wed
3: thurs
4: fri (most common)
5: sat
6: sun
"""

movie_df["release_date.day-of-week"] -= 1

Years, months (represented numerically), dates, and days-of-week (represented like above) are integers, not floats. We want our data to reflect that. 

In [41]:
import pandas as pd
import numpy as np

# Replace non-finite values with NaN, and convert to integers where possible
movie_df[["release_date.year", "release_date.month", "release_date.day-of-month", "release_date.day-of-week"]] = (
    movie_df[["release_date.year", "release_date.month", "release_date.day-of-month", "release_date.day-of-week"]]
    .astype('Int64')  # Convert to nullable integer type (preserves NaN)
)


Python has a native list type, but CSVs do not! The creator of this database elected to represent lists, instead of [a, b, c], like a|b|c. This is fine, but we want to change it so that we can work with them in Python.

In [42]:
# Convert "categories" nominated/won from "|"-delimited strings to lists of strings
for column in movie_df.columns:
    if "categories" in column:
        # Convert to list if string, if already list then keep as list, otherwise empty list (no nominations)
        movie_df[column] = movie_df[column].apply(lambda entry: entry.split("|") if isinstance(entry, str) else (entry if isinstance(entry, list) else []))

# Sanity check: make sure each entry length matches (for oscar at least)
oscar_mismatch_rows = movie_df[movie_df['Oscar_nominated_categories'].apply(len) != movie_df['Oscar_nominated']]
oscar_mismatch_rows[["Oscar_nominated_categories", "Oscar_nominated"]]

,Oscar_nominated_categories,Oscar_nominated


In [43]:
# convert "genre" from "|"-delimited strings into lists of strings
assert movie_df["genre"].isna().sum() == 0 # luckily, every movie has a genre already

# convert to list if string, if already list then keep as list, otherwise empty list (should never happen)
movie_df["genre"] = movie_df["genre"].apply(lambda entry: entry.split("|") if isinstance(entry, str) else (entry if isinstance(entry, list) else []))

assert all (movie_df["genre"].apply(lambda x: len(x) > 0 and isinstance(x, list))) # confirm all entries still have a genre and that all are lists

movie_df["genre"]

0           [Comedy, Fantasy, Romance]
1       [Animation, Adventure, Comedy]
2          [Action, Adventure, Family]
3          [Biography, Drama, Romance]
4          [Adventure, Drama, Fantasy]
                     ...              
1178                  [Drama, Romance]
1179                  [Drama, Romance]
1180       [Biography, Drama, History]
1181                    [Crime, Drama]
1182       [Biography, Drama, History]
Name: genre, Length: 1183, dtype: object

Now that the easy part of the preprocessing is complete, we can move on to the more difficult part. Actually filling in missing information. 
- Missing release dates
- Missing popularity
- Missing grosses
- Missing reviews (users, critics)
- Missing rating (G, PG-13, R, ...)

Some movies have missing release dates. We can find out how many below. 

In [ ]:
# Get movies without entered release dates
movie_df[movie_df["release_date"].isna()]["movie"]

1045                                     Mudbound
1060                                       Wonder
1165                                    Lady Bird
1169    Three Billboards Outside Ebbing, Missouri
1173                                     Marshall
1177                          The Florida Project
1178                         Call Me by Your Name
1179                               Phantom Thread
1181                        Roman J. Israel, Esq.
Name: movie, dtype: object

Since it's so few, we can just manually fill in the data from IMDB to mimic the original database as closely as possible. 

In [45]:
# Then we manually fix these using IMDB data, matching the original database. 
movie_release_dict = {
    "Mudbound": pd.to_datetime("2017-11-17", yearfirst=True, dayfirst=False),
    "Wonder": pd.to_datetime("2017-11-17", yearfirst=True, dayfirst=False),
    "Lady Bird": pd.to_datetime("2017-12-1", yearfirst=True, dayfirst=False),
    "Three Billboards Outside Ebbing, Missouri": pd.to_datetime("2017-12-1", yearfirst=True, dayfirst=False),
    "Marshall": pd.to_datetime("2017-10-13", yearfirst=True, dayfirst=False),
    "The Florida Project": pd.to_datetime("2017-11-10", yearfirst=True, dayfirst=False),
    "Call Me by Your Name": pd.to_datetime("2018-1-19", yearfirst=True, dayfirst=False),
    "Phantom Thread": pd.to_datetime("2018-1-19", yearfirst=True, dayfirst=False),
    "Roman J. Israel, Esq.": pd.to_datetime("2017-11-22", yearfirst=True, dayfirst=False),
}

# fill missing release dates
no_release_date = movie_df["release_date"].isna()
movie_df.loc[no_release_date, "release_date"] = movie_df["movie"].map(movie_release_dict)

# Extract year, month, and day only for the rows in no_release_date
movie_df.loc[no_release_date, "release_date.year"] = movie_df["release_date"].dt.year
movie_df.loc[no_release_date, "release_date.month"] = movie_df["release_date"].dt.month
movie_df.loc[no_release_date, "release_date.day-of-month"] = movie_df["release_date"].dt.day

# NOTE: datetime package uses 0 as Monday, 6 as Sunday. CSV uses 1 as Monday and 7 as Sunday.  
# That's why we changed to datetime package's way earlier! 
# When adding new weekdays,we can just use datetime's dt.weekday.
movie_df.loc[movie_df["release_date.day-of-week"].isna(), "release_date.day-of-week"] = movie_df["release_date"].dt.weekday

movie_df[["release_date", "release_date.year", "release_date.month", "release_date.day-of-month", "release_date.day-of-week"]]

,release_date,release_date.year,release_date.month,release_date.day-of-month,release_date.day-of-week
0,2001-12-25,2001,12,25,1
1,2000-06-23,2000,6,23,4
2,2005-07-08,2005,7,8,4
3,2002-11-22,2002,11,22,4
4,2001-12-19,2001,12,19,2
...,...,...,...,...,...
1178,2018-01-19,2018,1,19,4
1179,2018-01-19,2018,1,19,4
1180,2017-10-06,2017,10,6,4
1181,2017-11-22,2017,11,22,2


A movie's *popularity* is an IMDB-specific statistic that's based on when the movie was scraped. 1 is the most popular movie on IMDB at the time, and a higher number means less popular. Because the score depends on when the data was scraped, we can't manually fill in missing data. Additionally, Moana being there suggests that we're not dealing with *super* unpopular movies. 
For this project, we're going to ignore popularity for those reasons.  

In [ ]:
print(sum(movie_df["popularity"].isna()))
print(movie_df[movie_df["popularity"].isna()][["movie"]])

Some films are missing reviews and metacores, but thankfully the number is quite small. We have over a thousand movies, remember. 
**Un**luckily, we can't fill these stats in now since more reviews (especially from users) have come in since scraping. This is another IMDB specific stat. 

In [ ]:
# movies without user reviews also have no critic reviews!
print(movie_df[movie_df["user_reviews"].isna()][["movie", "user_reviews"]]) # 7 missing
print(movie_df[movie_df["critic_reviews"].isna()][["movie", "critic_reviews"]]) # 7 missing

# ...
print(movie_df[movie_df["metascore"].isna()][["movie", "metascore"]]) # 14 missing, disjoint from previous group

10 movies do not have a "rating" entry. For 8 of these, we can manually fill it in from IMDB. (Not Rated is different from not having a rating at all.)

In [ ]:
# Manually assign ratings (luckily these are all considered "Not Rated".)
movie_df.loc[movie_df["movie_id"] == "tt0382330", "certificate"] = "Not Rated"
movie_df.loc[movie_df["movie_id"] == "tt0426578", "certificate"] = "Not Rated"
movie_df.loc[movie_df["movie_id"] == "tt0879843", "certificate"] = "Not Rated"
movie_df.loc[movie_df["movie_id"] == "tt1077262", "certificate"] = "Not Rated"
movie_df.loc[movie_df["movie_id"] == "tt1337193", "certificate"] = "Not Rated"
movie_df.loc[movie_df["movie_id"] == "tt2852406", "certificate"] = "Not Rated"
movie_df.loc[movie_df["movie_id"] == "tt3170902", "certificate"] = "Not Rated"
movie_df.loc[movie_df["movie_id"] == "tt4239726", "certificate"] = "Not Rated"

movie_df[movie_df["certificate"].isna()][["movie", "movie_id", "year", "certificate"]]

# Twilight Samurai and Milk of Sorrow are not classified as "Not Rated", their rating truly does not exist.


Lastly, some movies are missing US & Canada box office grosses. Thankfully, a movie's gross does not change very much after it's released, since its no longer being played in the theaters! For that reason, we felt comfortable manually filling in the grosses that were available on IMDB. Some remain unavailable.  

In [ ]:
# fill in missing grosses manually
# grosses are found on IMDB's Gross US & Canada rounded to the nearest $10k

gross_fill_dict = { # manually built from IMDB website (imdb.com)
    'Lagaan: Once Upon a Time in India': 909043,
    # 'Ken Park': 0, (missing US & Canada data)
    'Zus & zo': 49468,
    'The Twilight Samurai': 559765,
    'The Room': 549602,
    # 'Yesterday': 0, (missing US & Canada data)
    # 'Unthinkable': 0, (missing US & Canada data)
    # 'Angus, Thongs and Perfect Snogging': 0, (missing US & Canada data)
    # 'Death Proof': 0, (missing US & Canada data)
    # "Hachi: A Dog's Tale": 0, (missing US & Canada data)
    'Revanche': 258388,
    'Outside the Law': 96933,
    'Secret in Their Eyes': 20180155,
    'The Great Wall': 45540830,
    # 'Mudbound': 0, (missing US & Canada data)
    'Leviathan': 1092800,
    'Omar': 356000,
    'Trumbo': 7857741,
    'The Autopsy of Jane Doe': 10474,
    'Elle': 2341534,
    'Land of Mine': 435266,
    # 'The Girl with All the Gifts': 0, (missing US & Canada data)
    'Split': 138291365,
    # 'Jim: The James Foley Story': 0 (missing US & Canada data)
}
# round to $10k
gross_fill_dict = {key: round(value, -4) for key, value in gross_fill_dict.items()}


movie_df.loc[movie_df["gross"].isna(), "gross"] = movie_df["movie"].map(gross_fill_dict)

assert sum(movie_df["gross"].isna()) == 9

movie_df[movie_df["gross"].isna()][["year", "movie", "gross"]]

Lastly, we want to note that for several awards providers, there is missing data in the `_categories` columns. These are: 
- BAFTA
- LA Film Critics Association
- Austin Film Critics Association
- Hollywood Film
Thankfully, our analysis will focus on Oscars, but it's important to make sure when performing data analysis that you know what kind of data is there and what kind of data is missing. 

In [ ]:
# Numerous '_categories' entries are missing. Too numerous to add manually.
# However, the '_won' and '_nominated' *non '_category'* columns appear accurate following manual investigation.

# Bafta
movies_with_zero_length_bafta_element = movie_df[movie_df["BAFTA_nominated_categories"].apply(lambda x: any(len(elem) == 0 for elem in x))]
movies_with_zero_length_bafta_element[["movie", "BAFTA_won", "BAFTA_won_categories", "BAFTA_nominated", "BAFTA_nominated_categories"]]
print(len(movies_with_zero_length_bafta_element))

# LAFC
movies_with_zero_length_lafc_element = movie_df[movie_df["Los_Angeles_Film_Critics_Association_nominated_categories"].apply(lambda x: any(len(elem) == 0 for elem in x))]
movies_with_zero_length_lafc_element[["movie", "Los_Angeles_Film_Critics_Association_won", "Los_Angeles_Film_Critics_Association_won_categories", "Los_Angeles_Film_Critics_Association_nominated", "Los_Angeles_Film_Critics_Association_nominated_categories"]]
print(len(movies_with_zero_length_lafc_element))

# Austin
movies_with_zero_length_austin_element = movie_df[movie_df["Austin_Film_Critics_Association_nominated_categories"].apply(lambda x: any(len(elem) == 0 for elem in x))]
movies_with_zero_length_austin_element[["movie", "Austin_Film_Critics_Association_won", "Austin_Film_Critics_Association_won_categories", "Austin_Film_Critics_Association_nominated", "Austin_Film_Critics_Association_nominated_categories"]]
print(len(movies_with_zero_length_austin_element))

# Hollywood
movies_with_zero_length_hollywood_element = movie_df[movie_df["Hollywood_Film_nominated_categories"].apply(lambda x: any(len(elem) == 0 for elem in x))]
movies_with_zero_length_hollywood_element[["movie", "Hollywood_Film_won", "Hollywood_Film_won_categories", "Hollywood_Film_nominated", "Hollywood_Film_nominated_categories"]]
print(len(movies_with_zero_length_hollywood_element))

Finally, we can view our summary statistics now that all our data is ready for analysis. 

In [ ]:
movie_df.describe() # Convenient method on dataframes

## Exploratory Data Analysis

## Primary Analysis

## Visualization

## Insights & Conclusions